In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics import silhouette_samples

# Estilo dos gráficos: grade suave e cores consistentes (igual aos dias da aula).
plt.style.use("ggplot")

# Carregar o mesmo dataset usado no material de apoio e nos dias 2 e 5.
# na_values=["null"] -> trata string "null" no CSV como valor ausente (NaN).
df = pd.read_csv("/kaggle/input/datasets/yashdevladdha/uber-ride-analytics-dashboard/ncr_ride_bookings.csv", na_values=["null"])

# Limpeza básica de IDs (remover aspas extras que às vezes vêm no CSV).
df["Booking ID"] = df["Booking ID"].str.replace('"', '', regex=False)
df["Customer ID"] = df["Customer ID"].str.replace('"', '', regex=False)

# Converter data e hora para tipo datetime (permite extrair hora, dia da semana, etc.).
df["Date"] = pd.to_datetime(df["Date"])
df["Time"] = pd.to_datetime(df["Time"], format="%H:%M:%S")

# Só usamos corridas COMPLETADAS – são as que têm valor (Booking Value) e distância.
df_ok = df[df["Booking Status"] == "Completed"].copy().reset_index(drop=True)

# Selecionar apenas as colunas numéricas que vamos usar no clustering
# (removemos linhas com NaN nessas colunas para não quebrar o algoritmo).
colunas_cluster = ["Ride Distance", "Booking Value", "Avg VTAT", "Avg CTAT"]
df_cluster = df_ok[colunas_cluster].dropna().copy().reset_index(drop=True)

print(f"Dataset: {len(df):,} corridas no total")
print(f"Corridas completadas (com valor e distância): {len(df_cluster):,}")
print(f"Variáveis usadas no clustering: {colunas_cluster}")

In [ ]:
scaler = StandardScaler()

# .fit_transform(df_cluster) -> calcula média e desvio de cada coluna e já transforma os dados.
X = scaler.fit_transform(df_cluster)

# X é uma matriz numérica (array): cada linha = uma corrida, cada coluna = variável normalizada.
print("Dados normalizados (primeiras 5 linhas):")
print(pd.DataFrame(X, columns=colunas_cluster).head())
print("\nCada coluna agora tem média = 0 e desvio = 1.")

In [ ]:
k_escolhido = 3

kmeans = KMeans(n_clusters=k_escolhido, random_state=42, n_init=10)
kmeans.fit(X)

# .labels_ -> array com o número do cluster de cada linha (0, 1, 2, ...).
rotulos = kmeans.labels_
df_cluster["cluster"] = rotulos

print(f"K-Means aplicado com K = {k_escolhido}")
print("Tamanho de cada grupo:")
print(df_cluster["cluster"].value_counts().sort_index())



In [ ]:
# Testar K de 2 até 10 e guardar a inércia de cada modelo.
K_range = range(2, 11)
inercias = []

for k in K_range:
    # n_clusters=k -> número de grupos.
    # random_state=42 -> mesma semente para resultado reproduzível.
    # n_init=10 -> algoritmo roda 10 vezes com centros iniciais diferentes e fica com o melhor.
    modelo_k = KMeans(n_clusters=k, random_state=42, n_init=10)
    modelo_k.fit(X)
    inercias.append(modelo_k.inertia_)

# Gráfico do cotovelo: eixo X = K, eixo Y = inércia.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(K_range, inercias, "o-", linewidth=2, markersize=8)
ax.set_xlabel("Número de clusters (K)")
ax.set_ylabel("Inércia")
ax.set_title("Método do cotovelo - qual K escolher?\n(procurar onde a curva 'dobra')")
ax.set_xticks(K_range)
plt.tight_layout()
plt.show()

# Em muitos datasets reais o 'cotovelo' não é muito nítido.
# K=3 ou K=4 costumam ser escolhas razoáveis para explicar em aula.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Cores distintas para cada cluster (paleta Set1 do seaborn).
cores = sns.color_palette("Set1", k_escolhido)

for i in range(k_escolhido):
    mask = df_cluster["cluster"] == i

    ax.scatter(
        df_cluster.loc[mask, "Ride Distance"],
        df_cluster.loc[mask, "Booking Value"],
        c=[cores[i]],
        label=f"Cluster {i}",
        alpha=0.5,
        s=15,
    )

ax.set_xlabel("Distância da corrida (km)")
ax.set_ylabel("Valor da corrida (R$)")
ax.set_title("Clusters de corridas — Distância x Valor\n(cada cor = um grupo encontrado pelo K-Means)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# kmeans.cluster_centers_ -> matriz: cada linha = centro de um cluster (em escala normalizada).
centros_normalizados = kmeans.cluster_centers_
centros_originais = scaler.inverse_transform(centros_normalizados)

# Criar um DataFrame com os centros nas colunas originais (para legenda e tabela).
df_centros = pd.DataFrame(centros_originais, columns=colunas_cluster)
df_centros["cluster"] = range(k_escolhido)

fig, ax = plt.subplots(figsize=(10, 6))

for i in range(k_escolhido):
    mask = df_cluster["cluster"] == i
    ax.scatter(
        df_cluster.loc[mask, "Ride Distance"],
        df_cluster.loc[mask, "Booking Value"],
        c=[cores[i]],
        label=f"Cluster {i}",
        alpha=0.4,
        s=12,
    )

# Marcar os centros com X grande e preto (fácil de ver na explicação).
ax.scatter(
    df_centros["Ride Distance"],
    df_centros["Booking Value"],
    marker="X",
    s=300,
    c="black",
    edgecolors="white",
    linewidths=2,
    label="Centro do cluster",
    zorder=5,
)

ax.set_xlabel("Distância da corrida (km)")
ax.set_ylabel("Valor da corrida (R$)")
ax.set_title("Clusters e seus centros (X)\n(o algoritmo agrupa pontos próximos a cada X)")
ax.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:


medias = df_cluster.groupby("cluster")[colunas_cluster].mean().round(2)
medias["quantidade"] = df_cluster.groupby("cluster").size()

print("\n=== MÉDIA POR CLUSTER (resultado do K-Means) ===\n")
print(medias)



In [ ]:
sil_media = silhouette_score(X, kmeans.labels_)
db_score = davies_bouldin_score(X, kmeans.labels_)

print("=== MÉTRICAS DE CLUSTERING ===")
print(f"Silhueta (média): {sil_media:.4f}")
print(f"Davies-Bouldin: {db_score:.4f}")

In [ ]:
sil_amostras = silhouette_samples(X, kmeans.labels_)
df_cluster["silhueta"] = sil_amostras

plt.figure(figsize=(8,4))

plt.hist(
    sil_amostras,
    bins=40,
    color="steelblue",
    edgecolor="white"
)

plt.axvline(
    sil_media,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Média = {sil_media:.3f}"
)

plt.title("Distribuição da silhueta")
plt.xlabel("Coeficiente de silhueta")
plt.ylabel("Frequência")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
K_range_aval = range(2, 11)

inercias_aval = []
silhuetas_aval = []
db_aval = []

for k in K_range_aval:
    
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    km.fit(X)
    
    inercias_aval.append(km.inertia_)
    silhuetas_aval.append(silhouette_score(X, km.labels_))
    db_aval.append(davies_bouldin_score(X, km.labels_))


fig, axes = plt.subplots(1, 3, figsize=(14,4))

# Elbow (Inertia)
axes[0].plot(K_range_aval, inercias_aval, "bo-")
axes[0].set_title("Inertia (Elbow)")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia")

# Silhouette
axes[1].plot(K_range_aval, silhuetas_aval, "go-")
axes[1].set_title("Silhouette")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Score")

# Davies-Bouldin
axes[2].plot(K_range_aval, db_aval, "ro-")
axes[2].set_title("Davies-Bouldin")
axes[2].set_xlabel("K")
axes[2].set_ylabel("Score")

plt.tight_layout()
plt.show()

In [ ]:
tamanhos = df_cluster.groupby("cluster").size()
props = (tamanhos / len(df_cluster) * 100).round(1)

print("=== TAMANHO POR CLUSTER ===")
print(tamanhos)

print("\n=== PROPORÇÃO POR CLUSTER (%) ===")
print(props)

In [ ]:
print("\n=== CONCLUSÃO DO CLUSTERING ===")

print(f"K testados: {list(K_range_aval)}")
print(f"Melhor Silhouette observado: {max(silhuetas_aval):.4f}")
print(f"Menor Davies-Bouldin observado: {min(db_aval):.4f}")

k_melhor = K_range_aval[silhuetas_aval.index(max(silhuetas_aval))]

print(f"\nSugestão de K ideal (baseado na Silhouette): {k_melhor}")

print("\nInterpretação:")
print("- A métrica Silhouette indica separação moderada entre clusters.")
print("- Davies-Bouldin sugere que os clusters não estão fortemente sobrepostos.")
print("- A análise conjunta das métricas sugere um número de clusters próximo do melhor Silhouette.")
print("- O modelo de clustering é aceitável para segmentação exploratória, mas não apresenta separação forte.")